In [1]:
import os
import numpy as np
import glob
from collections import defaultdict
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gsw
from pyproj import Geod
import pandas as pd
import warnings


# Data Import

In [11]:
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="Not a valid ID")

excluding_models = ["CESM2" , 'MPI-ESM1-2-LR', 'FGOALS-g3', 'CanESM5-1', 'GISS-E2-2-G', 'NorESM2-LM']# models not included in calculation
#excluding_models = ['MPI-ESM1-2-HR']
models = []

def dataconcat(scenario, variable):
    dir_path = f"/glade/work/stevenxu/AMOC_models/{variable}/scenarios/{scenario}"
    all_files = glob.glob(os.path.join(dir_path, "*.nc"))

    groups = defaultdict(list)
    for fp in all_files:
        fname = os.path.basename(fp)
        model_name = fname.split("_")[2]
        groups[model_name].append(fp)

    datasets = {}
    for prefix, files in groups.items():
        if prefix in excluding_models:
            continue

        files = sorted(files)
        print(f"Testing files for {prefix}")

        clean_files = []
        for f in files:
            # skip zero-byte files immediately
            if os.path.getsize(f) == 0:
                print(f"  Skipping empty file for {prefix}: {f}")
                continue
            try:
                xr.open_dataset(f, engine="netcdf4").close()
                clean_files.append(f)
            except Exception as e:
                print(f"  Skipping unreadable file for {prefix}: {f}")
                print("    Error:", repr(e))

        if not clean_files:
            print(f"⚠ No valid files left for {prefix}, skipping model.")
            continue

        print(f"Concatenating {len(clean_files)} valid files for {prefix}")
        ds = xr.open_mfdataset(
            clean_files,
            combine="by_coords",
            parallel=True,
            engine="netcdf4",
            use_cftime=True,
        )
        datasets[prefix] = ds
        models.append(prefix)

    return datasets
              
sst_datasets = dataconcat("PIControl", "sea_surface_temperature") 
sss_datasets = dataconcat("PIControl", "sea_surface_salinity") 
hf_datasets = dataconcat("PIControl", "heatflux") 
wf_datasets = dataconcat("PIControl", "waterflux")
models = set(models)

Testing files for MRI-ESM2-0
Concatenating 1 valid files for MRI-ESM2-0
Testing files for ICON-ESM-LR
Concatenating 50 valid files for ICON-ESM-LR
Testing files for CanESM5
Concatenating 5 valid files for CanESM5
Testing files for ACCESS-ESM1-5
Concatenating 3 valid files for ACCESS-ESM1-5
Testing files for GISS-E2-1-G-CC
Concatenating 4 valid files for GISS-E2-1-G-CC
Testing files for FGOALS-f3-L
Concatenating 5 valid files for FGOALS-f3-L
Testing files for CAS-ESM2-0
Concatenating 6 valid files for CAS-ESM2-0
Testing files for MIROC6
Concatenating 8 valid files for MIROC6
Testing files for ACCESS-CM2
Concatenating 1 valid files for ACCESS-CM2
Testing files for GISS-E2-1-G-CC
Concatenating 4 valid files for GISS-E2-1-G-CC
Testing files for ICON-ESM-LR
Concatenating 50 valid files for ICON-ESM-LR
Testing files for FGOALS-f3-L
Concatenating 5 valid files for FGOALS-f3-L
Testing files for MPI-ESM1-2-HR
Concatenating 100 valid files for MPI-ESM1-2-HR
Testing files for MIROC6
Concatenating

In [6]:
def subtract_years_cftime(t, years):
    """Return a new cftime object with years subtracted (same month/day)."""
    return type(t)(
        t.year - years, t.month, t.day,
        t.hour, t.minute, t.second, t.microsecond,
        has_year_zero=t.has_year_zero
    )

def align_time(model, lastNyear):
    # Make sure the model exists everywhere
    for name, dct in [
        ("sst", sst_datasets),
        ("sss", sss_datasets),
        ("hf",  hf_datasets),
        ("wf",  wf_datasets),
    ]:
        if model not in dct:
            print(f"Skipping {model}: missing in {name}_datasets")
            return

    end_times = []
    for variable_ds in [sst_datasets, sss_datasets, hf_datasets, wf_datasets]:
        end_times.append(variable_ds[model]['time'].isel(time=-1).values.item())

    start_time = subtract_years_cftime(min(end_times), lastNyear)
    end_time   = min(end_times)
    print(f"Aligning time for {model} to last {lastNyear} years: {start_time} to {end_time}")

    sst_datasets[model] = sst_datasets[model].sel(time=slice(start_time, end_time))
    sss_datasets[model] = sss_datasets[model].sel(time=slice(start_time, end_time))
    hf_datasets[model]  = hf_datasets[model].sel(time=slice(start_time, end_time))
    wf_datasets[model]  = wf_datasets[model].sel(time=slice(start_time, end_time))


for model in models:
    align_time(model, 20)

Aligning time for MIROC6 to last 20 years: 3679-12-16 12:00:00 to 3699-12-16 12:00:00
Aligning time for MRI-ESM2-0 to last 20 years: 2530-12-16 12:00:00 to 2550-12-16 12:00:00
Aligning time for ICON-ESM-LR to last 20 years: 4479-12-16 12:00:00 to 4499-12-16 12:00:00
Aligning time for CAS-ESM2-0 to last 20 years: 0529-12-16 12:00:00 to 0549-12-16 12:00:00
Aligning time for ACCESS-ESM1-5 to last 20 years: 0980-12-16 12:00:00 to 1000-12-16 12:00:00
Aligning time for GISS-E2-1-G-CC to last 20 years: 1994-12-16 12:00:00 to 2014-12-16 12:00:00
Aligning time for FGOALS-f3-L to last 20 years: 1079-12-16 12:00:00 to 1099-12-16 12:00:00
Aligning time for ACCESS-CM2 to last 20 years: 1429-12-16 12:00:00 to 1449-12-16 12:00:00
Aligning time for CanESM5 to last 20 years: 6180-12-16 12:00:00 to 6200-12-16 12:00:00


# Calculate sea surface density

In [13]:
def compute_surface_density(model, sst_datasets, sss_datasets, last_n_months=None):
    T  = sst_datasets[model]['tos']          
    SP = sss_datasets[model]['sos']          
    # addubg vertices for accurate area-weighted calculations for next step
    #vertices_latitude = sst_datasets[model]["vertices_latitude"]
    #vertices_longitude = sst_datasets[model]["vertices_longitude"]

    p0 = 0.0

    if last_n_months is not None:
        T  = T.isel(time=slice(-last_n_months, None))
        SP = SP.isel(time=slice(-last_n_months, None))

    """SA = xr.apply_ufunc(
        gsw.SA_from_SP, SP, p0, lon, lat,
        dask='parallelized', vectorize=True, output_dtypes=[float]
    )
    CT = xr.apply_ufunc(
        gsw.CT_from_t, SA, T, p0,
        dask='parallelized', vectorize=True, output_dtypes=[float]
    )"""
    alpha = xr.apply_ufunc(
        gsw.density.alpha, SP, T, p0,
        dask='parallelized', output_dtypes=[float]
    )
    beta = xr.apply_ufunc(
        gsw.density.beta, SP, T, p0,
        dask='parallelized', output_dtypes=[float]
    )
    rho = xr.apply_ufunc(
        gsw.density.rho, SP, T, p0,
        dask='parallelized', vectorize=True, output_dtypes=[float]
    )

    rho  = rho.rename('rho').assign_attrs(units='kg m-3',  long_name='Sea-surface density')
    alpha  = alpha.rename('alpha')
    beta  = beta.rename('beta')

    return xr.Dataset(dict(rho=rho, alpha=alpha, beta=beta))

#surf_den_ACCESS = compute_surface_density("NorESM2-LM", sst_datasets, sss_datasets, last_n_months=240)
#surf_den_ACCESS

In [16]:
sst_datasets["MIROC6"]

<xarray.Dataset> Size: 32GB
Dimensions:             (time: 9600, bnds: 2, y: 256, x: 360, vertices: 4)
Coordinates:
  * time                (time) object 77kB 3200-01-16 12:00:00 ... 3999-12-16...
  * y                   (y) float64 2kB -88.0 -85.75 -85.25 ... 150.5 152.4
  * x                   (x) float64 3kB 0.5 1.5 2.5 3.5 ... 357.5 358.5 359.5
    latitude            (y, x) float32 369kB dask.array<chunksize=(256, 360), meta=np.ndarray>
    longitude           (y, x) float32 369kB dask.array<chunksize=(256, 360), meta=np.ndarray>
Dimensions without coordinates: bnds, vertices
Data variables:
    time_bnds           (time, bnds) object 154kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    y_bnds              (time, y, bnds) float64 39MB dask.array<chunksize=(1200, 256, 2), meta=np.ndarray>
    x_bnds              (time, x, bnds) float64 55MB dask.array<chunksize=(1200, 360, 2), meta=np.ndarray>
    vertices_latitude   (time, y, x, vertices) float32 14GB dask.array<chunksize=(1200, 256, 360, 4), meta=np.ndarray>
    vertices_longitude  (time, y, x, vertices) float32 14GB dask.array<chunksize=(1200, 256, 360, 4), meta=np.ndarray>
    tos                 (time, y, x) float32 4GB dask.array<chunksize=(1, 256, 360), meta=np.ndarray>
Attributes: (12/44)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   0.0
    branch_time_in_parent:  365242.0
    creation_date:          2018-11-30T09:53:29Z
    ...                     ...
    title:                  MIROC6 output prepared for CMIP6
    variable_id:            tos
    variant_label:          r1i1p1f1
    license:                CMIP6 model data produced by MIROC is licensed un...
    cmor_version:           3.3.2
    tracking_id:            hdl:21.14100/c5d853f0-d006-4ced-867f-de2a32fe05c5

# Calculate F surf

In [14]:
def compute_fsurf(model,
                  sst_datasets, sss_datasets, hf_datasets, wf_datasets,
                  cp=3990.0, rho0=1027.0, rho_fw=1000.0, S0=35.0,
                  last_n_months=None):

    HF = hf_datasets[model]['hfds']  # W m^-2, 
    WF = wf_datasets[model]['wfo']     # kg m^-2 s^-1, 

    density_data = compute_surface_density(model, sst_datasets, sss_datasets, last_n_months=last_n_months)
    rho = density_data['rho']
    alpha = density_data['alpha']
    beta = density_data['beta']

    if last_n_months is not None:
        HF  = HF.isel(time=slice(-last_n_months, None))
        WF = WF.isel(time=slice(-last_n_months, None))

    # f_surf = -(alpha/cp) * f_heat  - (rho0/rho_fw) * beta * S0 * f_water
    fsurf = (alpha / cp) * HF  +  (rho0 / rho_fw) * beta * S0 * WF
    fsurf = fsurf.assign_attrs(
        long_name="Buoyancy-relevant surface forcing (Eq. 5)",
        description="(alpha/cp)*f_heat + (rho0/rho_fw)*beta*S0*f_water",
        units="",
        cp=cp, rho0=rho0, rho_fw=rho_fw, S0=S0
    )

    heat_comp = (alpha / cp) * HF
    fw_comp = (rho0 / rho_fw) * beta * S0 * WF

    return xr.Dataset(dict(fsurf=fsurf, rho=rho, heat_comp=heat_comp, fw_comp=fw_comp))

Fsurf_data = compute_fsurf(
    "CanESM5",
    sst_datasets=sst_datasets,
    sss_datasets=sss_datasets,
    hf_datasets=hf_datasets,
    wf_datasets=wf_datasets,
    last_n_months=240
)

Fsurf_datasets = {}
for model in models:
    print(f"Computing F_surf for {model}...")
    Fsurf_datasets[model] = compute_fsurf(
        model,
        sst_datasets=sst_datasets,
        sss_datasets=sss_datasets,
        hf_datasets=hf_datasets,
        wf_datasets=wf_datasets,
         last_n_months=240
    )

Computing F_surf for MIROC6...
Computing F_surf for MRI-ESM2-0...
Computing F_surf for ICON-ESM-LR...
Computing F_surf for CAS-ESM2-0...
Computing F_surf for ACCESS-ESM1-5...
Computing F_surf for GISS-E2-1-G-CC...
Computing F_surf for FGOALS-f3-L...
Computing F_surf for ACCESS-CM2...
Computing F_surf for CanESM5...
Computing F_surf for MPI-ESM1-2-HR...


KeyError: 'MPI-ESM1-2-HR'

In [17]:
ij_models = []
xy_models = []
i_models = []
norm_models = []

for model_name, ds in Fsurf_datasets.items():
    dims = set(ds.dims)

    if {'i', 'j'}.issubset(dims):
        ij_models.append(model_name)
    elif {'x', 'y'}.issubset(dims):
        xy_models.append(model_name)
    elif 'i' in dims and 'j' not in dims:
        i_models.append(model_name)
    elif {'lat', 'lon'}.issubset(dims):
        norm_models.append(model_name)

# Calculating Fgen at single timepoint

### Create density class list
Take max and min in rho values, and slice with interval 0.01

In [121]:
rho_min = 1015
rho_max = 1030

In [122]:
step_size = 0.05
rho_classes = np.arange(rho_min - step_size, rho_max + step_size, step_size)

### Dataset for area at each grid cell

In [7]:
dir_path = "/glade/work/stevenxu/AMOC_models/areacello"
all_files = glob.glob(os.path.join(dir_path, "*.nc")) 

area_ds = defaultdict(list)
model_names = []
for fp in all_files: 
    fname = os.path.basename(fp) 
    model_name = fname.split("_")[2] 
    area  = xr.open_dataset(fp)["areacello"]
    model_names.append(model_name)
    area_ds[model_name].append(area)
	

Use ACCESS one for missing areacello

In [8]:
area_ds['FGOALS-f3-L'] = [da.copy(deep=True) for da in area_ds['ACCESS-CM2']]

In [19]:
area_ds.keys()

dict_keys(['NorESM2-LM', 'CESM2', 'CanESM5', 'ICON-ESM-LR', 'MPI-ESM1-2-HR', 'ACCESS-ESM1-5', 'CAS-ESM2-0', 'GISS-E2-1-G-CC', 'GISS-E2-2-G', 'MIROC6', 'ACCESS-CM2', 'CanESM5-1', 'FGOALS-f3-L'])

Align areacello and actual data (180,288 -> 90, 144)

In [125]:
fs = Fsurf_datasets["GISS-E2-1-G-CC"]
area0 = area_ds["GISS-E2-1-G-CC"][0]          # (lat:180, lon:288)

# 1) coarsen to (lat:90, lon:144) — use SUM for area
area_coarse = area0.coarsen(lat=2, lon=2, boundary="trim").sum()

# 2) align to Fsurf grid (safe even if tiny coord diffs)
area_coarse = area_coarse.sel(
    lat=fs["lat"],
    lon=fs["lon"],
    method="nearest"
)

# optional: sanity check
print("Fsurf grid:", fs.sizes["lat"], fs.sizes["lon"])
print("Area  grid:", area_coarse.sizes["lat"], area_coarse.sizes["lon"])

# 3) save it for later use
area_ds["GISS-E2-1-G-CC"][0] = area_coarse

Fsurf grid: 90 144
Area  grid: 90 144


### Integration

Group by density intervals and adding up the area-weighted fsurf

In [126]:
Fgen_dict = {}

#### (i) models

In [127]:
for model, Fsurf_data in Fsurf_datasets.items():
    if model not in i_models:
        continue

    filtered_Fsurf = (
        Fsurf_data
        .where(Fsurf_data["latitude"] > 45, drop=True)
    )#.isel(time=slice(9, 10))

    fsurf = filtered_Fsurf["fsurf"]
    heat_comp = filtered_Fsurf["heat_comp"]
    fw_comp = filtered_Fsurf["fw_comp"]
    rho = filtered_Fsurf["rho"]

    area1 = area_ds[model][0].sel(i=filtered_Fsurf["i"])  # <-- critical alignment

    timepoints = filtered_Fsurf["time"].values

    weighted_fsurf = fsurf * area1
    weighted_heat_comp = heat_comp * area1
    weighted_fw_comp = fw_comp * area1

    # numpy area
    area_v = area1.values

    rows = []  # faster than Fgen.loc in loop
    for time in range(len(timepoints)):
        # pull numpy once per time (avoids dask/xarray objects in pandas)
        rho_t = rho.isel(time=time).values
        wf_t  = weighted_fsurf.isel(time=time).values
        wh_t  = weighted_heat_comp.isel(time=time).values
        wfw_t = weighted_fw_comp.isel(time=time).values

        for rhoclass in rho_classes:
            rhobot = rhoclass
            rhotop = rhoclass + step_size

            idx = np.where((rho_t > rhobot) & (rho_t < rhotop))[0]

            fgen_value = float(np.nansum(wf_t[idx])  / step_size / 1e6)
            heat_value = float(np.nansum(wh_t[idx])  / step_size / 1e6)
            fw_value   = float(np.nansum(wfw_t[idx]) / step_size / 1e6)
            area_sum   = float(np.nansum(area_v[idx]))

            rows.append([time, rhoclass + step_size/2, fgen_value, heat_value, fw_value, area_sum])

    Fgen = pd.DataFrame(rows, columns=["time", "rho", "Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"])
    Fgen = Fgen.groupby("rho", as_index=False)[["Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"]].mean()

    Fgen_dict[model] = Fgen
    print(f"Completed Fgen calculation for {model}")

KeyboardInterrupt: 

#### (i, j) models

In [ ]:
for model, Fsurf_data in Fsurf_datasets.items():
    if model not in ij_models:
        continue

    # -------------------------
    # 1) Build mask (j,i) and get kept points (MultiIndex labels)
    # -------------------------
    mask = (Fsurf_data["latitude"] > 45)

    mask_s = mask.stack(points=("j", "i"))  # MultiIndex points=(j,i)
    keep_pts = mask_s.where(mask_s, drop=True).coords["points"]

    # -------------------------
    # 2) Stack fields and select kept points (still lazy xarray)
    # -------------------------
    fsurf_s = Fsurf_data["fsurf"].stack(points=("j", "i")).sel(points=keep_pts)
    heat_s  = Fsurf_data["heat_comp"].stack(points=("j", "i")).sel(points=keep_pts)
    fw_s    = Fsurf_data["fw_comp"].stack(points=("j", "i")).sel(points=keep_pts)
    rho_s   = Fsurf_data["rho"].stack(points=("j", "i")).sel(points=keep_pts)

    # areacello (pick the first one if your area_ds stores a list per model)
    area1 = area_ds[model][0]
    area_s = area1.stack(points=("j", "i")).sel(points=keep_pts)

    # -------------------------
    # 3) Apply weights (still xarray), then convert to NumPy once
    #    Keep as (time, points) for speed (no need to unstack!)
    # -------------------------
    wf_s  = (fsurf_s * area_s)
    wh_s  = (heat_s  * area_s)
    wfw_s = (fw_s    * area_s)

    # OPTIONAL: restrict time for testing (matches your i-model testing)
    """wf_s  = wf_s.isel(time=slice(0, 1))
    wh_s  = wh_s.isel(time=slice(0, 1))
    wfw_s = wfw_s.isel(time=slice(0, 1))
    rho_s = rho_s.isel(time=slice(0, 1))"""

    timepoints = rho_s["time"].values

    # Convert to numpy arrays ONCE (prevents pandas/dask recursion issues)
    wf_np   = wf_s.values      # shape (time, points)
    wh_np   = wh_s.values
    wfw_np  = wfw_s.values
    rho_np  = rho_s.values     # density at points
    area_np = area_s.values    # shape (points,)

    # -------------------------
    # 4) Main loops (time x rho_bins), vectorized across points
    # -------------------------
    rows = []
    for t in range(len(timepoints)):
        rho_t = rho_np[t, :]

        for rhoclass in rho_classes:
            rhobot = rhoclass
            rhotop = rhoclass + step_size

            idx = np.where((rho_t > rhobot) & (rho_t < rhotop))[0]

            fgen_value  = float(np.nansum(wf_np[t, idx])  / step_size / 1e6)
            heat_value  = float(np.nansum(wh_np[t, idx])  / step_size / 1e6)
            fw_value    = float(np.nansum(wfw_np[t, idx]) / step_size / 1e6)
            area_sum    = float(np.nansum(area_np[idx]))

            rows.append([t, rhoclass + step_size/2, fgen_value, heat_value, fw_value, area_sum])

    Fgen = pd.DataFrame(rows, columns=["time", "rho", "Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"])
    Fgen = Fgen.groupby("rho", as_index=False)[["Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"]].mean()

    Fgen_dict[model] = Fgen
    print(f"Completed Fgen calculation for {model}")

Completed Fgen calculation for CanESM5
Completed Fgen calculation for ACCESS-CM2
Completed Fgen calculation for ACCESS-ESM1-5
Completed Fgen calculation for FGOALS-f3-L


#### (y, x) models

In [ ]:
for model, Fsurf_data in Fsurf_datasets.items():
    if model not in xy_models: # 
        continue

    # -------------------------
    # 1) Build mask (y,x) and get kept points (MultiIndex labels)
    # -------------------------
    mask = (Fsurf_data["latitude"] > 45)

    mask_s = mask.stack(points=("y", "x"))  # MultiIndex points=(y,x)
    keep_pts = mask_s.where(mask_s, drop=True).coords["points"]

    # -------------------------
    # 2) Stack fields and select kept points (still lazy xarray)
    # -------------------------
    fsurf_s = Fsurf_data["fsurf"].stack(points=("y", "x")).sel(points=keep_pts)
    heat_s  = Fsurf_data["heat_comp"].stack(points=("y", "x")).sel(points=keep_pts)
    fw_s    = Fsurf_data["fw_comp"].stack(points=("y", "x")).sel(points=keep_pts)
    rho_s   = Fsurf_data["rho"].stack(points=("y", "x")).sel(points=keep_pts)

    # areacello (pick the first one if your area_ds stores a list per model)
    area1 = area_ds[model][0]
    area_s = area1.stack(points=("y", "x")).sel(points=keep_pts)

    # -------------------------
    # 3) Apply weights (still xarray), then convert to NumPy once
    #    Keep as (time, points) for speed (no need to unstack!)
    # -------------------------
    wf_s  = (fsurf_s * area_s)
    wh_s  = (heat_s  * area_s)
    wfw_s = (fw_s    * area_s)

    # OPTIONAL: restrict time for testing (matches your i-model testing)
    """wf_s  = wf_s.isel(time=slice(0, 1))
    wh_s  = wh_s.isel(time=slice(0, 1))
    wfw_s = wfw_s.isel(time=slice(0, 1))
    rho_s = rho_s.isel(time=slice(0, 1))"""

    timepoints = rho_s["time"].values

    # Convert to numpy arrays ONCE (prevents pandas/dask recursion issues)
    wf_np   = wf_s.values      # shape (time, points)
    wh_np   = wh_s.values
    wfw_np  = wfw_s.values
    rho_np  = rho_s.values     # density at points
    area_np = area_s.values    # shape (points,)

    # -------------------------
    # 4) Main loops (time x rho_bins), vectorized across points
    # -------------------------
    rows = []
    for t in range(len(timepoints)):
        rho_t = rho_np[t, :]

        for rhoclass in rho_classes:
            rhobot = rhoclass
            rhotop = rhoclass + step_size

            idx = np.where((rho_t > rhobot) & (rho_t < rhotop))[0]

            fgen_value  = float(np.nansum(wf_np[t, idx])  / step_size / 1e6)
            heat_value  = float(np.nansum(wh_np[t, idx])  / step_size / 1e6)
            fw_value    = float(np.nansum(wfw_np[t, idx]) / step_size / 1e6)
            area_sum    = float(np.nansum(area_np[idx]))

            rows.append([t, rhoclass + step_size/2, fgen_value, heat_value, fw_value, area_sum])

    Fgen = pd.DataFrame(rows, columns=["time", "rho", "Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"])
    Fgen = Fgen.groupby("rho", as_index=False)[["Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"]].mean()

    Fgen_dict[model] = Fgen
    print(f"Completed Fgen calculation for {model}")

#### (lat, lon) models

In [ ]:
for model, Fsurf_data in Fsurf_datasets.items():
    if model not in norm_models:
        continue

    filtered_Fsurf = (
        Fsurf_data
        .where(Fsurf_data["lat"] > 45, drop=True)
    )#.isel(time=slice(9, 13))
    

    fsurf = filtered_Fsurf["fsurf"]
    heat_comp = filtered_Fsurf["heat_comp"]
    fw_comp = filtered_Fsurf["fw_comp"]
    rho = filtered_Fsurf["rho"]

    lat_idx = np.where(Fsurf_data["lat"].values > 45)[0]
    area1 = area_ds[model][0].isel(lat=lat_idx)

    timepoints = filtered_Fsurf["time"].values

    #weighted_fsurf = fsurf * area1
    #weighted_heat_comp = heat_comp * area1
    #weighted_fw_comp = fw_comp * area1

    # numpy area
    area_v = area1.values

    rows = []
    for time in range(len(timepoints)):
        rho_t = rho.isel(time=time).values          # (lat, lon)
        fsurf_t = fsurf.isel(time=time).values
        heat_comp_t = heat_comp.isel(time=time).values
        fw_comp_t = fw_comp.isel(time=time).values
        wf_t  = fsurf_t * area_v
        wh_t  = heat_comp_t * area_v
        wfw_t = fw_comp_t * area_v

        print(time)

        for rhoclass in rho_classes:
            rhobot = rhoclass
            rhotop = rhoclass + step_size

            m = (rho_t > rhobot) & (rho_t < rhotop)   # boolean (lat, lon)
            

            fgen_value = float(np.nansum(wf_t[m])  / step_size / 1e6)
            heat_value = float(np.nansum(wh_t[m])  / step_size / 1e6)
            fw_value   = float(np.nansum(wfw_t[m]) / step_size / 1e6)
            area_sum   = float(np.nansum(area_v[m]))

            rows.append([time, rhoclass + step_size/2, fgen_value, heat_value, fw_value, area_sum])

    Fgen = pd.DataFrame(rows, columns=["time", "rho", "Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"])
    Fgen = Fgen.groupby("rho", as_index=False)[["Fgen", "HeatFlux", "FreshwaterFlux", "AreaSum"]].mean()

    Fgen_dict[model] = Fgen
    print(f"Completed Fgen calculation for {model}")

0
1
2
3
Completed Fgen calculation for CAS-ESM2-0
0
1
2
3
Completed Fgen calculation for GISS-E2-1-G-CC


In [114]:
import pickle

save_path = "/glade/work/stevenxu/AMOC_models/Fgen_Allmodels.pkl"
with open(save_path, "wb") as f:
    pickle.dump(Fgen_dict, f)